# Notebook 4: Trading Signals & Backtesting
Generate trading signals, run walk-forward backtest, and evaluate strategy performance.

## 1. Setup

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})
from src.features.feature_pipeline import load_features
data = load_features('brent')
test_df = data['test_df']
feature_cols = data['feature_cols']
print(f"Test period: {test_df.index[0].date()} → {test_df.index[-1].date()}")
print(f"Test rows: {len(test_df):,}")

## 2. Load Model & Generate Signals

In [ ]:
from pathlib import Path
import joblib

model_path = Path('../models/xgboost_brent.joblib')
if model_path.exists():
    from src.models.xgboost_model import XGBoostOilForecaster
    model = XGBoostOilForecaster.load(model_path)
    dir_probs = model.predict_direction_proba(test_df)
    print(f"Model loaded. Generating signals for {len(test_df)} trading days...")
else:
    print("Model not found. Run: python src/models/train.py")
    print("Using synthetic signal probabilities for demonstration...")
    rng = np.random.default_rng(42)
    dir_probs = np.clip(0.5 + test_df['roc_5d'].fillna(0)*3 + rng.normal(0, 0.1, len(test_df)), 0.15, 0.85)

signals = pd.Series(
    ['BUY' if p >= 0.58 else 'SELL' if p <= 0.42 else 'HOLD' for p in dir_probs],
    index=test_df.index
)
print(f"\nSignal distribution:")
print(signals.value_counts())
print(f"\nBuy signals:  {(signals=='BUY').sum()} ({(signals=='BUY').mean():.1%})")
print(f"Sell signals: {(signals=='SELL').sum()} ({(signals=='SELL').mean():.1%})")
print(f"Hold signals: {(signals=='HOLD').sum()} ({(signals=='HOLD').mean():.1%})")

## 3. Signal Accuracy Analysis

In [ ]:
# How accurate were BUY signals? Did price actually go up next day?
signal_df = pd.DataFrame({
    'price':     test_df['close'],
    'signal':    signals,
    'prob_up':   dir_probs,
    'next_ret':  test_df['close'].pct_change().shift(-1),
    'went_up':   (test_df['close'].pct_change().shift(-1) > 0).astype(int),
})

print("Signal accuracy breakdown:")
for sig in ['BUY', 'SELL', 'HOLD']:
    subset = signal_df[signal_df['signal'] == sig]
    if len(subset) == 0: continue
    if sig == 'BUY':
        accuracy = (subset['went_up'] == 1).mean()
    elif sig == 'SELL':
        accuracy = (subset['went_up'] == 0).mean()
    else:
        accuracy = 0.5
    avg_ret = subset['next_ret'].mean() * 100
    print(f"  {sig:<6}: {len(subset):>4} signals | accuracy={accuracy:.1%} | avg 1d return={avg_ret:+.3f}%")

# Calibration curve: does P(up) = actual frequency of up moves?
bins = np.linspace(0.2, 0.85, 8)
actual_freq = []
for i in range(len(bins)-1):
    mask = (signal_df['prob_up'] >= bins[i]) & (signal_df['prob_up'] < bins[i+1])
    if mask.sum() > 10:
        actual_freq.append((bins[i]+bins[i+1])/2)
    
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0.2, 0.85], [0.2, 0.85], 'k--', alpha=0.4, label='Perfect calibration')
prob_bins = pd.cut(signal_df['prob_up'], bins=bins)
cal = signal_df.groupby(prob_bins)['went_up'].agg(['mean', 'count'])
cal = cal[cal['count'] > 10]
ax.scatter(cal.index.map(lambda x: x.mid), cal['mean'],
           s=cal['count']/5, color='#378ADD', alpha=0.7)
ax.set_xlabel('Predicted P(up)')
ax.set_ylabel('Actual frequency of up moves')
ax.set_title('Probability Calibration Curve')
ax.legend()
plt.tight_layout()
plt.savefig('../docs/images/signal_calibration.png', bbox_inches='tight')
plt.show()

## 4. Walk-Forward Backtest

In [ ]:
from src.signals.backtester import WalkForwardBacktester

bt = WalkForwardBacktester(
    initial_capital=1_000_000,
    transaction_cost_pct=0.0005,
    position_size_pct=0.10,
    stop_loss_atr_mult=2.0,
    take_profit_atr_mult=3.0,
)

if 'model' in dir() and model is not None:
    trades_df, equity_series, bt_signals = bt.run(test_df, feature_cols, model)
else:
    # Simulate trades using signal series
    trades_list = []
    equity = [1_000_000]
    for i in range(1, len(test_df)-1):
        price = test_df['close'].iloc[i]
        signal_val = signals.iloc[i]
        ret = test_df['close'].pct_change().iloc[i+1]
        if signal_val == 'BUY':
            pnl = 100_000 * ret - 50
            trades_list.append({'date': test_df.index[i], 'pnl': pnl, 'side': 'LONG', 'type': 'CLOSE'})
        elif signal_val == 'SELL':
            pnl = -100_000 * ret - 50
            trades_list.append({'date': test_df.index[i], 'pnl': pnl, 'side': 'SHORT', 'type': 'CLOSE'})
        equity.append(equity[-1] + trades_list[-1]['pnl'] if trades_list and signals.iloc[i] != 'HOLD' else equity[-1])
    trades_df = pd.DataFrame(trades_list)
    equity_series = pd.Series(equity[:len(test_df)], index=test_df.index[:len(equity)])

metrics = bt.compute_metrics(trades_df, equity_series, test_df['close'])
print("\nBacktest Results:")
print(f"{'─'*45}")
for k, v in metrics.items():
    print(f"  {k:<35} {v}")

## 5. Performance Chart

In [ ]:
fig = plt.figure(figsize=(15, 10))
gs  = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.3)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, :])
ax3 = fig.add_subplot(gs[2, 0])
ax4 = fig.add_subplot(gs[2, 1])

# Price + signals
ax1.plot(test_df.index, test_df['close'], color='#888', linewidth=0.8)
for sig, color, marker in [('BUY','#1D9E75','^'), ('SELL','#E24B4A','v')]:
    mask = signals == sig
    if mask.any():
        ax1.scatter(test_df.index[mask], test_df['close'][mask],
                   marker=marker, color=color, s=30, zorder=5, label=sig, alpha=0.7)
ax1.set_title('Brent Crude — Signals Overlay')
ax1.set_ylabel('Price ($/bbl)')
ax1.legend(fontsize=8)

# Equity curve
bh = test_df['close'] / test_df['close'].iloc[0] * 1_000_000
ax2.plot(equity_series.index, equity_series, color='#1D9E75', linewidth=1.5, label='Strategy')
ax2.plot(bh.index, bh, '--', color='#888', linewidth=1.0, alpha=0.7, label='Buy & Hold')
ax2.set_title('Portfolio Equity vs Buy & Hold')
ax2.set_ylabel('Value ($)')
ax2.legend()
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e6:.2f}M'))

# Trade PnL distribution
if len(trades_df) > 0 and 'pnl' in trades_df.columns:
    pnls = trades_df['pnl'].dropna()
    ax3.hist(pnls, bins=min(25, len(pnls)),
             color=['#1D9E75' if p > 0 else '#E24B4A' for p in pnls],
             alpha=0.8, edgecolor='white')
    ax3.axvline(pnls.mean(), color='#EF9F27', linewidth=1.5, linestyle='--',
                label=f'Mean: ${pnls.mean():,.0f}')
    ax3.set_title('Trade PnL Distribution')
    ax3.legend()

# Drawdown
rolling_max = equity_series.cummax()
drawdown = (equity_series - rolling_max) / rolling_max * 100
ax4.fill_between(drawdown.index, drawdown, 0, color='#E24B4A', alpha=0.4)
ax4.set_title(f"Drawdown (Max: {metrics.get('max_drawdown_pct',0):.1f}%)")
ax4.set_ylabel('Drawdown (%)')

strat_ret  = metrics.get('total_return_pct', 0)
bh_ret     = metrics.get('buy_hold_return_pct', 0)
sharpe     = metrics.get('sharpe_ratio', 0)
n_trades   = metrics.get('n_trades', 0)
win_rate   = metrics.get('win_rate_pct', 0)
fig.suptitle(
    f"Backtest Summary — Strategy: {strat_ret:.1f}% | B&H: {bh_ret:.1f}% | "
    f"Sharpe: {sharpe:.2f} | Trades: {n_trades} | Win rate: {win_rate:.1f}%",
    fontsize=11, y=0.98
)
plt.savefig('../docs/images/backtest_report.png', bbox_inches='tight')
plt.show()

## 6. Monthly Return Heatmap

In [ ]:
monthly_returns = equity_series.resample('M').last().pct_change().dropna() * 100

if len(monthly_returns) >= 12:
    pivot = pd.DataFrame({
        'year': monthly_returns.index.year,
        'month': monthly_returns.index.month,
        'return': monthly_returns.values
    }).pivot(index='year', columns='month', values='return')

    import seaborn as sns
    fig, ax = plt.subplots(figsize=(12, max(3, len(pivot))))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
                center=0, ax=ax, cbar_kws={'label': 'Return (%)'},
                linewidths=0.3)
    ax.set_xlabel('Month')
    ax.set_ylabel('Year')
    ax.set_title('Monthly Strategy Returns (%)')
    month_names = ['Jan','Feb','Mar','Apr','May','Jun',
                   'Jul','Aug','Sep','Oct','Nov','Dec']
    ax.set_xticklabels([month_names[int(x.get_text())-1]
                        for x in ax.get_xticklabels()], rotation=0)
    plt.tight_layout()
    plt.savefig('../docs/images/monthly_returns.png', bbox_inches='tight')
    plt.show()
else:
    print(f"Only {len(monthly_returns)} months of test data — increase test window for heatmap")

## Next Steps: Production
```bash
# Start inference API
uvicorn src.serving.api:app --reload --port 8000

# Start analyst dashboard
streamlit run dashboard/app.py

# Run drift monitoring
python src/monitoring/drift_detector.py

# Full Docker stack
cd docker && docker-compose up --build
```